In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.special import gammaln

plt.rcParams.update({
    "text.usetex":        False,
    "font.family":        "serif",
    "font.serif":         ["Palatino", "Palatino Linotype", "Georgia",
                           "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset":   "stix",
    "axes.labelsize":     11,
    "axes.titlesize":     11,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "figure.dpi":         150,
})

WEBGREEN   = (0,       0.5,    0    )
ITALIC_RED = (177/255, 56/255, 69/255)

def _blend(rgb, mix):
    return tuple(c + (1 - c) * mix for c in rgb)

def simulate_erw(n, p, q=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    xi = np.empty(n + 1, dtype=np.int8)
    xi[0] = 0
    xi[1] = 1 if rng.random() < q else -1
    for t in range(1, n):
        past = xi[rng.integers(0, t) + 1]
        xi[t + 1] = past if rng.random() < p else -past
    S = np.empty(n + 1, dtype=np.int64)
    S[0] = 0
    S[1:] = np.cumsum(xi[1:])
    return S

def exact_second_moment(n_arr, p):

    alpha = 2*p - 1
    log_ratio = gammaln(n_arr + 2*alpha) - gammaln(n_arr + 1) - gammaln(2*alpha)
    return n_arr / (2*alpha - 1) * (np.exp(log_ratio) - 1)

Q       = 0.5
N_STEPS = 1000
N_MEAN  = 800

PANELS = [
    (0.80, WEBGREEN,   r"$p = 0.80$", 15),
    (0.90, ITALIC_RED, r"$p = 0.90$", 15),
]

rng        = np.random.default_rng(42)
steps_plot = np.arange(0, N_STEPS + 1)
steps_th   = np.arange(1, N_STEPS + 1, dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(5.0, 3.0), sharey=False)
fig.subplots_adjust(left=0.10, right=0.97, bottom=0.16, top=0.88, wspace=0.46)

for ax, (p_val, base_col, title, n_thin) in zip(axes, PANELS):
    thin_paths = [simulate_erw(N_STEPS, p=p_val, q=Q, rng=rng).astype(float)**2
                  for _ in range(n_thin)]
    mat  = np.array([simulate_erw(N_STEPS, p=p_val, q=Q, rng=rng).astype(float)**2
                     for _ in range(N_MEAN)])
    emp  = mat.mean(axis=0)

    exact = exact_second_moment(steps_th, p_val)

    light = _blend(base_col, 0.60)
    darkd = tuple(c * 0.62 for c in base_col)

    for S2 in thin_paths:
        ax.plot(steps_plot, S2, color=light, lw=0.55, alpha=0.75, rasterized=True)
    ax.plot(steps_plot, emp,
            color=darkd, lw=0.9, alpha=0.9, zorder=3, solid_capstyle="round")
    ax.plot(steps_th, exact,
            color=darkd, lw=0.9, alpha=0.9, zorder=4,
            ls=(0, (5, 3)), solid_capstyle="round")

    ymax = max(emp[-1], exact[-1]) * 1.3
    ax.set_xlim(0, N_STEPS)
    ax.set_ylim(0, ymax)
    ax.set_title(title, fontsize=9.5, pad=5, color="darkgray")
    ax.set_xlabel(r"$n$", labelpad=3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.55)
    ax.spines["bottom"].set_linewidth(0.55)
    ax.tick_params(width=0.55, length=3, direction="out")
    ax.xaxis.set_major_locator(ticker.FixedLocator([0, N_STEPS // 2, N_STEPS]))
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"${int(x)}$"))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4, integer=True))
    if ax is axes[0]:
        ax.set_ylabel(r"$S_n^2$", labelpad=4)